# Clase 3 — Unidad 3: Trabajando con Datos Biomédicos: Manejo Básico de Datos

**Curso:** Introducción al Análisis de Datos — 2026, Segundo Semestre
**Material de referencia:** *Python Essentials for Biomedical Data Analysis* — Capítulo 3, "Working with Biomedical Data: Basic Data Handling"

## Objetivos de aprendizaje
- Reconocer los distintos **tipos y formatos** de datos biomédicos y los desafíos que presentan.
- Adquirir habilidades prácticas para **importar y exportar** datos en distintos formatos: archivos planos, datos jerárquicos, imágenes médicas y datos genómicos.
- Familiarizarse con el manejo de **conjuntos de datos grandes**.

## Contenidos de la clase
1. Panorama de los tipos de datos biomédicos
2. Desafíos en el manejo de datos biomédicos
3. Formatos de datos más usados
4. Importación de datos (archivos planos, JSON/XML, datos grandes, imágenes, genómica, transcriptómica)
5. Exportación de datos
6. Ejercicios de práctica

> Esta clase corresponde a la **Unidad 3** del libro guía. Para que todos los ejemplos funcionen en clase, primero **crearemos archivos de ejemplo** y luego los leeremos. Las celdas que dependen de librerías externas están protegidas con `try/except`, de modo que el notebook completo se pueda ejecutar sin errores.

## 1. Panorama de los tipos de datos biomédicos

Los datos biomédicos pueden presentarse en muchas formas. Conocer sus características es el primer paso para manejarlos correctamente.

| Tipo de dato | Descripción breve |
|---|---|
| **Genómicos** | Genes y secuencias de ADN. Conjuntos muy grandes (ej. secuenciación de genoma completo). |
| **Transcriptómicos** | Conjunto de transcritos de ARN; informan sobre función y regulación génica. |
| **Proteómicos** | Estudio del proteoma: expresión, estructura, función e interacciones de proteínas. |
| **Metabolómicos** | Metabolitos (moléculas pequeñas); reflejan cambios metabólicos asociados a enfermedades. |
| **Epigenéticos** | Cambios heredables en la expresión génica sin alterar la secuencia del ADN (ej. metilación). |
| **Microbioma** | Bacterias, virus y hongos que habitan el cuerpo; clave en inmunidad y salud digestiva. |
| **Ensayos clínicos** | Demografía de pacientes, tratamientos, resultados clínicos y efectos adversos. |
| **Registros clínicos (EHR)** | Historia clínica, diagnósticos, medicamentos, alergias, resultados de laboratorio. |
| **Imágenes médicas** | Resonancia (MRI), tomografía (CT), rayos X, ecografía. Archivos grandes. |
| **Biobancos** | Muestras biológicas (sangre, tejido, ADN) con datos clínicos asociados. |
| **Farmacogenómicos** | Relación entre genética y respuesta a fármacos; base de la medicina personalizada. |
| **Salud digital** | Datos de wearables e IoT: actividad física, sueño, frecuencia cardíaca. |

## 2. Desafíos en el manejo de datos biomédicos

Trabajar con datos biomédicos implica varios desafíos que pueden afectar la investigación, las decisiones clínicas y el desarrollo de fármacos:

- **Volumen y complejidad:** los conjuntos son grandes y heterogéneos (desde secuencias genéticas hasta ensayos clínicos) y requieren herramientas computacionales avanzadas.
- **Calidad e integridad:** datos precisos, consistentes y completos son indispensables; se necesita validación y limpieza rigurosas para evitar conclusiones erróneas.
- **Interoperabilidad:** integrar datos de fuentes y formatos distintos exige estandarización y protocolos de intercambio.
- **Privacidad y seguridad:** los datos suelen contener información personal sensible, sujeta a leyes de privacidad y estándares éticos.
- **Uso intensivo de recursos:** el procesamiento y almacenamiento demanda infraestructura y personal especializado.

## 3. Formatos de datos más usados

| Categoría | Formatos | Uso típico |
|---|---|---|
| **Archivos planos** | CSV, TSV | Datos tabulares simples; ideales para conjuntos pequeños/medianos. |
| **Datos jerárquicos** | JSON, XML | Datos anidados de experimentos o estudios complejos. |
| **Datos científicos a gran escala** | HDF5, Zarr | Grandes volúmenes heterogéneos; almacenamiento y acceso eficientes. |
| **Imágenes médicas** | DICOM | Estándar para imágenes médicas y sus metadatos. |
| **Datos genómicos** | FASTA, FASTQ | Secuencias de nucleótidos/aminoácidos; FASTQ agrega puntajes de calidad. |
| **Formatos propietarios** | .raw, .mae, .cdx | Específicos de instrumentos o software; requieren conversión. |

> **FASTA**: formato de texto para secuencias; cada secuencia empieza con una línea de encabezado que inicia con `>`.
> **FASTQ**: similar a FASTA, pero incluye un puntaje de calidad por cada nucleótido (útil en secuenciación de nueva generación, NGS).

## Preparación del entorno

Para que todos los ejemplos de importación funcionen, primero **crearemos archivos de ejemplo** (CSV, TSV, JSON y XML) en una carpeta llamada `datos_ejemplo`. Así podremos leerlos inmediatamente después.

También instalaremos **Pandas**, la librería central para manejar datos tabulares.

In [ ]:
%pip install pandas

In [ ]:
import os

# Carpeta donde guardaremos los archivos de ejemplo de esta clase
os.makedirs("datos_ejemplo", exist_ok=True)

# 1) Archivo CSV con datos de pacientes (separado por comas)
csv_contenido = """paciente_id,edad,sexo,glucosa,colesterol,diagnostico
P001,45,F,95,180,Sano
P002,53,M,140,220,Prediabetes
P003,38,F,88,175,Sano
P004,61,M,165,240,Diabetes
P005,29,F,92,160,Sano
"""
with open("datos_ejemplo/pacientes.csv", "w", encoding="utf-8") as f:
    f.write(csv_contenido)

# 2) Mismo contenido pero separado por tabulaciones (.tsv)
with open("datos_ejemplo/pacientes.tsv", "w", encoding="utf-8") as f:
    f.write(csv_contenido.replace(",", "\t"))

print("Archivos planos creados en 'datos_ejemplo':", os.listdir("datos_ejemplo"))

In [ ]:
import json

# 3) Archivo JSON con estructura jerárquica (los exámenes están anidados)
registros = [
    {"paciente_id": "P001", "edad": 45, "diagnostico": "Sano",
     "examenes": {"glucosa": 95, "colesterol": 180}},
    {"paciente_id": "P002", "edad": 53, "diagnostico": "Prediabetes",
     "examenes": {"glucosa": 140, "colesterol": 220}},
    {"paciente_id": "P003", "edad": 38, "diagnostico": "Sano",
     "examenes": {"glucosa": 88, "colesterol": 175}},
]
with open("datos_ejemplo/pacientes.json", "w", encoding="utf-8") as f:
    json.dump(registros, f, ensure_ascii=False, indent=2)

print("Archivo JSON creado.")

In [ ]:
# 4) Archivo XML con la misma información
xml_contenido = """<?xml version="1.0" encoding="UTF-8"?>
<pacientes>
    <paciente>
        <id>P001</id>
        <edad>45</edad>
        <diagnostico>Sano</diagnostico>
    </paciente>
    <paciente>
        <id>P002</id>
        <edad>53</edad>
        <diagnostico>Prediabetes</diagnostico>
    </paciente>
</pacientes>
"""
with open("datos_ejemplo/pacientes.xml", "w", encoding="utf-8") as f:
    f.write(xml_contenido)

print("Archivo XML creado. Contenido de la carpeta:", os.listdir("datos_ejemplo"))

## 4. Importación de datos

Veremos técnicas para importar distintos tipos de datos biomédicos a Python: archivos planos (CSV, TSV), datos jerárquicos (JSON, XML), grandes volúmenes, imágenes médicas y datos genómicos.

### 4.1 Lectura de archivos planos

Python trae funciones incorporadas (`open()` y el módulo `csv`) para leer archivos de texto. Sin embargo, para manipular datos de forma cómoda se usa la librería **Pandas**.

In [ ]:
# Lectura de un archivo de texto línea por línea con la función incorporada open()
with open("datos_ejemplo/pacientes.csv", "r", encoding="utf-8") as file:
    for line in file:
        print(line.strip())   # strip() elimina espacios y saltos de línea

In [ ]:
# Lectura de un CSV con el módulo csv (cada fila se lee como una lista)
import csv

with open("datos_ejemplo/pacientes.csv", "r", encoding="utf-8") as file:
    reader = csv.reader(file, delimiter=",")
    for row in reader:
        print(row)

**Con Pandas** la lectura es mucho más simple: `read_csv()` y `read_table()` devuelven un **DataFrame** (una tabla con filas y columnas, similar a una hoja de cálculo).

In [ ]:
import pandas as pd

# Leer un archivo CSV (separado por comas)
df = pd.read_csv("datos_ejemplo/pacientes.csv", sep=",")
print(df.head())

In [ ]:
# Leer un archivo separado por tabulaciones (TSV)
df_tsv = pd.read_csv("datos_ejemplo/pacientes.tsv", sep="\t")
print(df_tsv.head())

# read_table asume tabulación por defecto
df_tabla = pd.read_table("datos_ejemplo/pacientes.tsv")
print(df_tabla.head())

In [ ]:
# Pandas facilita explorar rápidamente los datos
print("Dimensiones (filas, columnas):", df.shape)
print("\nTipos de datos por columna:")
print(df.dtypes)
print("\nEstadísticas descriptivas de las columnas numéricas:")
print(df.describe())

### 4.2 Datos jerárquicos (JSON y XML)

Los formatos **JSON** y **XML** almacenan datos con estructura anidada. Python trae las librerías `json` y `xml.etree.ElementTree` para leerlos.

In [ ]:
import json

# Leer un archivo JSON
with open("datos_ejemplo/pacientes.json", "r", encoding="utf-8") as file:
    data = json.load(file)

print(data)

In [ ]:
import pandas as pd

# Convertir datos JSON con estructura anidada a un DataFrame.
# pd.json_normalize "aplana" los campos anidados (examenes.glucosa, examenes.colesterol).
df_json = pd.json_normalize(data)
print(df_json)

Para **XML** recorremos el árbol a partir de la raíz (`root`) y extraemos cada elemento.

In [ ]:
import xml.etree.ElementTree as ET

# Leer y recorrer un archivo XML
tree = ET.parse("datos_ejemplo/pacientes.xml")
root = tree.getroot()

for paciente in root:
    # Para cada paciente, construimos un diccionario {etiqueta: texto}
    print(paciente.tag, "->", {sub.tag: sub.text for sub in paciente})

In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET

# Convertir el XML a un DataFrame
tree = ET.parse("datos_ejemplo/pacientes.xml")
root = tree.getroot()

filas = []
for paciente in root:
    registro = {sub.tag: sub.text for sub in paciente}
    filas.append(registro)

df_xml = pd.DataFrame(filas)
print(df_xml)

### 4.3 Importar grandes volúmenes de datos

Cuando un conjunto de datos es demasiado grande para caber en memoria, se usan técnicas como:
- **Chunking** (por bloques): dividir el archivo en partes y procesarlas una a una.
- **Lazy loading** (carga perezosa): leer los datos solo cuando se necesitan, con funciones generadoras.
- **HDF5**: un formato binario eficiente para grandes volúmenes.

In [ ]:
import pandas as pd
import numpy as np

# Crear un archivo CSV más grande para practicar (50.000 filas)
n = 50_000
grande = pd.DataFrame({
    "paciente_id": np.arange(1, n + 1),
    "glucosa": np.random.randint(70, 200, size=n),
    "colesterol": np.random.randint(120, 280, size=n),
})
grande.to_csv("datos_ejemplo/laboratorio_grande.csv", index=False)
print("Archivo grande creado con", n, "filas.")

In [ ]:
# Chunking: leer el archivo grande por bloques (chunks) para no saturar la memoria
chunk_size = 10_000
total_filas = 0
suma_glucosa = 0

for chunk in pd.read_csv("datos_ejemplo/laboratorio_grande.csv", chunksize=chunk_size):
    total_filas += len(chunk)
    suma_glucosa += chunk["glucosa"].sum()

print("Filas procesadas:", total_filas)
print("Glucosa promedio:", round(suma_glucosa / total_filas, 2))

In [ ]:
# Lazy loading: leer un archivo línea por línea con una función generadora
def leer_archivo_grande(archivo):
    """Generador que lee un archivo línea por línea (ahorra memoria)."""
    while True:
        linea = archivo.readline()
        if not linea:
            break
        yield linea

# Mostrar solo las primeras 5 líneas del archivo grande
with open("datos_ejemplo/laboratorio_grande.csv", "r", encoding="utf-8") as f:
    for i, linea in enumerate(leer_archivo_grande(f)):
        print(linea.strip())
        if i >= 4:
            break

**HDF5** (con la librería `h5py`) es ideal para conjuntos muy grandes: permite compresión y acceso rápido. Requiere instalar `h5py` (`%pip install h5py`).

In [ ]:
# HDF5 usa la librería h5py y numpy. Si no está instalada: %pip install h5py
try:
    import h5py
    import numpy as np

    # Escribir un conjunto de datos en formato HDF5 (con compresión)
    datos = np.random.rand(1000, 50)   # 1000 muestras, 50 variables
    with h5py.File("datos_ejemplo/datos.h5", "w") as h5:
        h5.create_dataset("expresion", data=datos, compression="gzip")

    # Leer de vuelta el conjunto guardado
    with h5py.File("datos_ejemplo/datos.h5", "r") as h5:
        leidos = h5["expresion"][...]   # [...] carga todo el dataset como arreglo NumPy
        print("Forma del conjunto leído:", leidos.shape)
except ImportError:
    print("h5py no está instalado. Ejecute:  %pip install h5py")

### 4.4 Datos de imágenes médicas (DICOM)

**DICOM** es el estándar para imágenes médicas (contiene la imagen y metadatos como paciente, modalidad y fecha). Se lee con la librería `pydicom`. Otras librerías útiles son `SimpleITK` y `OpenCV`.

> El siguiente código es **ilustrativo**: requiere la librería `pydicom` y un archivo `.dcm` real (por ejemplo, del *Cancer Imaging Archive*). Está protegido con `try/except` para que no interrumpa la ejecución del notebook.

In [ ]:
# Lectura de imágenes DICOM con pydicom (requiere la librería y un archivo .dcm)
# Instalar con:  %pip install pydicom
try:
    import pydicom

    ds = pydicom.dcmread("datos_ejemplo/imagen.dcm")

    if hasattr(ds, "pixel_array"):
        imagen = ds.pixel_array          # datos de la imagen como arreglo NumPy
        print("Forma de la imagen:", imagen.shape)

    # Acceder a metadatos con .get() para evitar errores si falta el campo
    print("Paciente:", ds.get("PatientName", "N/A"))
    print("Modalidad:", ds.get("Modality", "N/A"))
    print("Fecha del estudio:", ds.get("StudyDate", "N/A"))
except ImportError:
    print("pydicom no está instalado. Ejecute:  %pip install pydicom")
except FileNotFoundError:
    print("No se encontró un archivo .dcm. Descargue uno para probar este ejemplo.")

### 4.5 Datos genómicos (FASTA / FASTQ)

**Biopython** permite leer formatos genómicos como FASTA (secuencias) y FASTQ (secuencias + calidad). Instalar con `%pip install biopython`.

Primero creamos un archivo FASTA de ejemplo y luego lo leemos.

In [ ]:
# Crear un archivo FASTA de ejemplo (siempre funciona, es texto plano)
fasta_texto = """>Secuencia1 Ejemplo de secuencia de ADN
ATGCGTACGTAGCTAGCTAGCTGACT
>Secuencia2 Otra secuencia de ADN
TTGGCCAATTGGCCAATTGGCCAA
"""
with open("datos_ejemplo/ejemplo.fasta", "w", encoding="utf-8") as f:
    f.write(fasta_texto)

# Leer el FASTA con Biopython
try:
    from Bio import SeqIO
    for seq_record in SeqIO.parse("datos_ejemplo/ejemplo.fasta", "fasta"):
        print(f"ID: {seq_record.id}")
        print(f"Secuencia: {seq_record.seq}")
        print(f"Longitud: {len(seq_record.seq)} nucleótidos\n")
except ImportError:
    print("Biopython no está instalado. Ejecute:  %pip install biopython")

In [ ]:
# Crear un archivo FASTQ de ejemplo (secuencia + puntajes de calidad)
fastq_texto = """@Lectura1
ATGCGTACGTAGCT
+
IIIIIIIIIIIIII
@Lectura2
TTGGCCAATTGGCC
+
FFFFFFFFFFFFFF
"""
with open("datos_ejemplo/ejemplo.fastq", "w", encoding="utf-8") as f:
    f.write(fastq_texto)

# Leer el FASTQ con Biopython (incluye los puntajes de calidad Phred)
try:
    from Bio import SeqIO
    for seq_record in SeqIO.parse("datos_ejemplo/ejemplo.fastq", "fastq"):
        print(f"ID: {seq_record.id}")
        print(f"Secuencia: {seq_record.seq}")
        print(f"Calidad (Phred): {seq_record.letter_annotations['phred_quality']}\n")
except ImportError:
    print("Biopython no está instalado. Ejecute:  %pip install biopython")

### 4.6 Datos transcriptómicos (AnnData)

Los datos transcriptómicos (RNA-seq, single-cell RNA-seq) suelen manejarse con **AnnData** ("Annotated Data"), que guarda juntos los datos y sus anotaciones:
- `.X`: la matriz de datos (típicamente conteos de expresión génica).
- `.obs`: metadatos de las observaciones (células o muestras).
- `.var`: información de las variables (genes).

Instalar con `%pip install anndata`.

In [ ]:
# Datos transcriptómicos con AnnData (requiere: %pip install anndata)
try:
    import anndata as ad
    import numpy as np

    matriz = np.random.rand(100, 10)                 # 100 células, 10 genes
    obs = {"tipo_celular": ["tipo1"] * 50 + ["tipo2"] * 50}
    var = {"gen": ["gen" + str(i) for i in range(10)]}

    adata = ad.AnnData(X=matriz, obs=obs, var=var)
    print("Forma de la matriz (.X):", adata.X.shape)
    print("Tipos celulares:", adata.obs["tipo_celular"].unique())
    print("Primeros genes:", adata.var["gen"].tolist()[:5], "...")
except ImportError:
    print("anndata no está instalado. Ejecute:  %pip install anndata")

## 5. Exportación de datos

Exportar datos es esencial para compartirlos, publicarlos o continuar el análisis en otras herramientas. Veremos cómo guardar datos en archivos planos, jerárquicos, a gran escala, imágenes y datos genómicos.

### 5.1 Exportar a archivos planos

In [ ]:
# Escribir texto plano con write()
with open("datos_ejemplo/salida.txt", "w", encoding="utf-8") as file:
    file.write("Creado con Python")

# Escribir varias líneas con writelines()
lineas = ["Primera línea\n", "Segunda línea\n"]
with open("datos_ejemplo/salida2.txt", "w", encoding="utf-8") as file:
    file.writelines(lineas)

print("Archivos de texto creados.")

In [ ]:
# Exportar a CSV con el módulo csv
import csv

datos = [["Nombre", "Edad"], ["Ana", 30], ["Luis", 25]]
with open("datos_ejemplo/salida.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerows(datos)

print("CSV escrito con el módulo csv.")

In [ ]:
# Exportar un DataFrame a CSV y TSV con Pandas (método to_csv)
df.to_csv("datos_ejemplo/pacientes_exportado.csv", index=False)
df.to_csv("datos_ejemplo/pacientes_exportado.tsv", sep="\t", index=False)

print("DataFrame exportado a CSV y TSV.")

### 5.2 Exportar datos jerárquicos (JSON, XML, pickle)

In [ ]:
import json

# Exportar un diccionario a JSON
registro = {"nombre": "Ana", "edad": 30, "diagnostico": "Sano"}
with open("datos_ejemplo/salida.json", "w", encoding="utf-8") as file:
    json.dump(registro, file, ensure_ascii=False, indent=2)

print("JSON exportado.")

In [ ]:
import xml.etree.ElementTree as ET

# Construir una estructura XML desde objetos de Python
root = ET.Element("Pacientes")
paciente = ET.SubElement(root, "Paciente")

nombre = ET.SubElement(paciente, "Nombre")
nombre.text = "Ana"
edad = ET.SubElement(paciente, "Edad")
edad.text = "30"

tree = ET.ElementTree(root)
with open("datos_ejemplo/salida.xml", "wb") as file:
    tree.write(file, encoding="utf-8", xml_declaration=True)

print("XML exportado.")

In [ ]:
import pickle

# pickle serializa objetos de Python (útil para guardarlos y recuperarlos idénticos)
datos = {"nombre": "Ana", "edad": 30, "tratamientos": ["dieta", "ejercicio"]}
with open("datos_ejemplo/salida.pkl", "wb") as file:
    pickle.dump(datos, file)

# Leer de vuelta el objeto
with open("datos_ejemplo/salida.pkl", "rb") as file:
    recuperado = pickle.load(file)

print("Objeto recuperado desde pickle:", recuperado)

### 5.3 Exportación a gran escala

Para conjuntos muy grandes conviene escribir por **bloques (chunks)**, de modo que la memoria solo contenga una parte a la vez. Con `mode="a"` cada bloque se **agrega** al mismo archivo.

In [ ]:
import os

# Evitar duplicar filas si la celda se ejecuta varias veces
ruta = "datos_ejemplo/exportado_por_partes.csv"
if os.path.exists(ruta):
    os.remove(ruta)

# Exportar el DataFrame grande por partes
chunk_size = 10_000
for i in range(0, len(grande), chunk_size):
    parte = grande.iloc[i:i + chunk_size]
    parte.to_csv(
        ruta,
        mode="a",                 # agregar al archivo
        index=False,
        header=(i == 0),          # encabezado solo en el primer bloque
    )

print("Exportación por partes completada.")

> El formato **HDF5** (visto en la sección 4.3) también sirve para exportar a gran escala usando `compression="gzip"`, lo que reduce el tamaño del archivo y acelera el acceso.

### 5.4 Exportar imágenes procesadas

Las imágenes procesadas suelen guardarse en formatos accesibles (JPEG, PNG, TIFF) con librerías como **PIL (Pillow)**, **OpenCV** o **matplotlib**.

In [ ]:
# Exportar una imagen procesada (ejemplo con datos simulados). Requiere: %pip install pillow
try:
    import numpy as np
    from PIL import Image

    imagen_data = (np.random.rand(256, 256) * 255).astype("uint8")  # imagen 256x256 en escala de grises
    imagen = Image.fromarray(imagen_data)
    imagen.save("datos_ejemplo/imagen_procesada.png")
    print("Imagen guardada como PNG con PIL.")
except ImportError:
    print("Pillow (PIL) no está instalado. Ejecute:  %pip install pillow")

### 5.5 Exportar datos genómicos

Con **Biopython** podemos escribir secuencias en formato FASTA (o FASTQ), preservando identificadores y descripciones.

In [ ]:
# Exportar una secuencia a formato FASTA con Biopython
try:
    from Bio import SeqIO
    from Bio.Seq import Seq
    from Bio.SeqRecord import SeqRecord

    registro = SeqRecord(
        Seq("ATGCGTACGTAGCTAGCTAGCTGACT"),
        id="Secuencia1",
        description="Secuencia de ADN de ejemplo",
    )
    with open("datos_ejemplo/exportado.fasta", "w") as file:
        SeqIO.write(registro, file, "fasta")
    print("Secuencia exportada a FASTA.")
except ImportError:
    print("Biopython no está instalado. Ejecute:  %pip install biopython")

## 6. Ejercicios de práctica

Usa los archivos de la carpeta `datos_ejemplo` (o crea los tuyos) para resolver:

1. **Leer un CSV:** importa `pacientes.csv` con Pandas y muestra solo las columnas `paciente_id`, `edad` y `diagnostico`.
2. **Filtrar datos:** a partir del DataFrame anterior, muestra solo los pacientes con `glucosa` mayor a 100.
3. **Trabajar con JSON:** carga `pacientes.json` y extrae, para cada paciente, su `paciente_id` y el valor de `glucosa` (campo anidado dentro de `examenes`).
4. **Importar un conjunto grande:** lee `laboratorio_grande.csv` en bloques (chunks) de 5.000 filas y calcula el **colesterol máximo** global.
5. **Leer datos genómicos:** crea un archivo FASTA con 2 secuencias e imprime el nombre y la longitud de cada una con Biopython.
6. **Exportar a archivo plano:** guarda el DataFrame de pacientes en un nuevo archivo `mis_pacientes.csv` sin la columna de índice.
7. **Exportar datos jerárquicos:** guarda un diccionario con datos de un paciente (incluyendo una lista de tratamientos) en un archivo JSON.
8. **Exportar a gran escala:** exporta `laboratorio_grande.csv` a un archivo nuevo escribiéndolo por partes (chunks).

### Preguntas para reflexionar
1. ¿Cuáles son los principales tipos de datos biomédicos y en qué se diferencian?
2. ¿Qué desafíos suelen aparecer al manejar datos biomédicos?
3. ¿Qué formatos comunes y propietarios se usan en datos biomédicos?
4. ¿Cómo importarías un conjunto de datos muy grande sin comprometer el rendimiento del sistema?
5. ¿Por qué conviene usar formatos como JSON o XML para datos jerárquicos?

In [ ]:
# Espacio para resolver los ejercicios
